# Ingesta incremental de mediciones a Bronze

Conservar los archivos en `recepcion/mediciones/lote_01/`, `lote_02/`, `lote_03/`, `lote_correccion/` y `lote_reenvio/`. Indicar el lote que se quiere comprobar en el widget. El notebook lee todas las subcarpetas con un único checkpoint persistente y solo añade los archivos nuevos.


In [24]:
import csv
import json
from pathlib import Path

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

config_path = next((parent / 'config' / 'bronze.json' for parent in (Path.cwd(), *Path.cwd().parents) if (parent / 'config' / 'bronze.json').is_file()), None)
if config_path is None:
    raise FileNotFoundError('No se encuentra config/bronze.json desde el directorio del notebook')
config = json.loads(config_path.read_text(encoding='utf-8'))
INPUT_DIR = config['mediciones_input_dir']
TARGET_TABLE = config['tables']['mediciones']
CHECKPOINT = config['checkpoints']['mediciones']
expected_columns = list(config['input_schemas']['mediciones'])
assert all(dtype.lower() == 'string' for dtype in config['input_schemas']['mediciones'].values())
SOURCE_SCHEMA = StructType([StructField(name, StringType(), True) for name in expected_columns])
print(INPUT_DIR, TARGET_TABLE, CHECKPOINT, sep='\n')


/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones
workspace.lensdata_bronze.mediciones
/Volumes/workspace/lensdata_bronze/archivos/checkpoints/bronze/mediciones


## Validación previa

La carpeta del lote solicitado debe existir y contener CSV. Se comprueban las cabeceras de todos los CSV visibles antes de escribir. No sobrescribir archivos ya ingeridos: las correcciones y los reenvíos llegan en archivos con nombres distintos.


In [25]:
def csv_files(directory):
    for entry in dbutils.fs.ls(directory):
        if entry.path.endswith("/"):
            yield from csv_files(entry.path)
        elif entry.path.lower().endswith(".csv"):
            yield entry.path

files = sorted(csv_files(INPUT_DIR))

assert files, "No hay archivos CSV"

for file_path in files:
    header_line = spark.read.text(file_path).first()[0].lstrip("\ufeff")
    actual_columns = next(csv.reader([header_line]))

    assert actual_columns == expected_columns, (
        f"Cabecera inesperada en {file_path}"
    )

print(f"Cabeceras verificadas: {len(files)}")
print(*files, sep="\n")

Cabeceras verificadas: 13
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_01/mediciones_2026-11-01.csv
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_02/mediciones_2026-11-02.csv
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-03.csv
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-04.csv
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-05.csv
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-06.csv
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-07.csv
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-08.csv
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-09.csv
/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones

## Carga

Las ocho columnas de origen siguen siendo cadenas. `_source_file` conserva la ruta de procedencia para las validaciones posteriores en Silver. Subida de información con Auto Loader, usando el formato `cloudFiles` 


In [34]:
mediciones = (spark.readStream.format('cloudFiles')
    .option('cloudFiles.format', 'csv')
    .option('header', 'true')
    .option('mode', 'FAILFAST')
    .option('ignoreLeadingWhiteSpace', 'false')
    .option('ignoreTrailingWhiteSpace', 'false')
    .option('cloudFiles.includeExistingFiles', 'true')
    .option('recursiveFileLookup', 'true')
    .schema(SOURCE_SCHEMA)
    .load(INPUT_DIR)
    .withColumn('_source_file', F.col('_metadata.file_path')))

query = (mediciones.writeStream.format('delta')
    .option('checkpointLocation', CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE))
query.awaitTermination()


In [28]:
resultado = spark.table(TARGET_TABLE)

print("Filas acumuladas:", resultado.count())

resultado.groupBy("_source_file").count().show(truncate=False)


Filas acumuladas: 471
+--------------------------------------------------------------------------------------------------+-----+
|_source_file                                                                                      |count|
+--------------------------------------------------------------------------------------------------+-----+
|/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-12.csv|39   |
|/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-04.csv|39   |
|/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-08.csv|39   |
|/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-06.csv|39   |
|/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-05.csv|39   |
|/Volumes/workspace/lensdata_bronze/archivos/recepcion/mediciones/lote_03/mediciones_2026-11-03.csv|39   |
|/Volumes/works

In [30]:
total_segunda = spark.table(TARGET_TABLE).count()

print("Registros anteriores:", total_inicial)
print("Registros actuales:", total_segunda)
print("Nuevos registros:", total_segunda - total_inicial)

Registros anteriores: 40
Registros actuales: 471
Nuevos registros: 431


## Prueba de idempotencia
Se ejecute de nuevo la carga con Auto Loader y ingiere de nuevo los registros ya cargados todo esto gracias al checkpoint cargado en la celda de carga. Esto no nos salva de alguna posible duplicación de archivos, de eso nos encargamos en la siguiente capa

In [33]:
total_antes = spark.table(TARGET_TABLE).count()
print("Total antes:", total_antes)

Total antes: 471


In [35]:
total_despues = spark.table(TARGET_TABLE).count()

assert total_antes == total_despues, "Se duplicaron registros"

print("Total antes:", total_antes)
print("Total después:", total_despues)
print("Nuevos registros:", total_despues - total_antes)
print("Idempotencia validada correctamente")

Total antes: 471
Total después: 471
Nuevos registros: 0
Idempotencia validada correctamente
